# How AI Changes Human Work
## Productivity, Reliance, and Skill Retention

**Empirical Analysis of 10,000 AI-Assisted Learning Sessions**  
**Period:** June 2024 – June 2025 | **N = 10,000 sessions**

---

### Research Questions
1. Does AI assistance level affect session efficiency?
2. Does AI assistance predict task success?
3. Does high AI reliance correlate with failure (overreliance)?
4. Which task types benefit most from AI?
5. Do beginners benefit more than experts?
6. Does satisfaction correlate with AI level and outcomes?
7. Can we predict struggle from session features?
8. What predicts willingness to use AI again?


## 0. Setup & Imports

In [ ]:
import sys
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
})

sys.path.insert(0, os.getcwd())
print('All imports successful.')


## 1. Data Loading & Cleaning

Steps: duplicate detection, range clamping, IQR×3 outlier winsorization, feature engineering.

In [ ]:
from src.data_cleaning import load_clean

df = load_clean('dataset.csv')
print(f'Dataset shape: {df.shape}')
df.head()


In [ ]:
df.describe(include='all')


In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'Total missing: {df.isnull().sum().sum()}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['AI_AssistanceLevel'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('AI Assistance Level Distribution')
df['FinalOutcome'].value_counts().plot(
    kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Final Outcome Distribution')
df['SatisfactionRating'].hist(ax=axes[2], bins=30,
    color='mediumseagreen', edgecolor='white')
axes[2].set_title('Satisfaction Rating Distribution')
plt.tight_layout()
plt.show()


## 2. Exploratory Data Analysis

In [ ]:
print('=== Student Level Counts ===')
print(df['StudentLevel'].value_counts())
print()
print('=== Discipline Counts ===')
print(df['Discipline'].value_counts())
print()
print('=== Task Type Counts ===')
print(df['TaskType'].value_counts())
print()
print('=== Date Range ===')
print(f"First: {df['SessionDate'].min().date()}  |  Last: {df['SessionDate'].max().date()}")


In [ ]:
# Session length and prompts by AI level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette('Blues_d', 5)
means = df.groupby('AI_AssistanceLevel')['SessionLengthMin'].mean()
sems  = df.groupby('AI_AssistanceLevel')['SessionLengthMin'].sem()
axes[0].bar(range(1,6), means, yerr=sems*1.96,
            color=palette, capsize=5, edgecolor='white')
axes[0].set_title('Mean Session Length by AI Level (+/-95% CI)')
axes[0].set_xlabel('AI Assistance Level')
axes[0].set_ylabel('Minutes')
axes[0].set_xticks(range(1,6))

df.boxplot(column='TotalPrompts', by='AI_AssistanceLevel', ax=axes[1],
           patch_artist=True, medianprops=dict(color='white', lw=2))
axes[1].set_title('Prompt Count by AI Level')
axes[1].set_xlabel('AI Assistance Level')
plt.suptitle('')
plt.tight_layout()
plt.show()


In [ ]:
success_overall = df['success'].mean()
print(f'Overall task success rate: {success_overall:.1%}')
print(f"Would use AI again: {df['UsedAgain'].mean():.1%}")
print()
print('Success rate by AI level:')
print(df.groupby('AI_AssistanceLevel')['success'].mean().round(3))
print()
print('Outcome breakdown:')
print(df['FinalOutcome'].value_counts(normalize=True).round(3))


## 3. Statistical Analysis

### RQ1: Session Efficiency

**Hypothesis:** Higher AI reliance leads to shorter sessions.

In [ ]:
from src.statistical_analysis import rq1_session_length

r1 = rq1_session_length(df)
print('=== RQ1: Session Length vs AI Level ===')
print('Group means (min):')
for lvl, mean in r1['group_means_session_min'].items():
    print(f'  Level {lvl}: {mean:.2f} min')
print()
print(f"ANOVA: F={r1['anova_f']}, p={r1['anova_p']} {r1['anova_sig']}")
tt = r1['ttest_low_vs_high']
print(f"Low AI mean: {tt['mean_1']:.2f}  |  High AI mean: {tt['mean_2']:.2f}")
print(f"Cohen's d = {tt['d']} ({tt['effect_label']})")
print()
print('Interpretation:', r1['interpretation'])


### RQ2: Task Success

In [ ]:
from src.statistical_analysis import rq2_task_success

r2 = rq2_task_success(df)
print(f"Overall success rate: {r2['overall_success_rate']:.1%}")
print(f"Chi-square: chi2({r2['dof']}) = {r2['chi2']}, p={r2['p_chi']}")
print(f"Cramer's V = {r2['cramers_v']}")
print()
print('Success rate by AI level:')
for lvl, rate in r2['success_rate_by_level'].items():
    print(f'  Level {lvl}: {rate:.1%}')
if r2.get('logistic'):
    lg = r2['logistic']
    print(f"OR (AI level): {lg['odds_ratio']}, p={lg['pval_AI_level']}")
print()
print('Interpretation:', r2['interpretation'])


In [ ]:
# Stacked bar: outcomes by AI level
pivot = df.groupby(['AI_AssistanceLevel', 'FinalOutcome']).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
ax = pivot_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
    color=['#2E8B57','#4682B4','#DAA520','#CD5C5C'], edgecolor='white')
ax.set_title('Outcome Distribution by AI Assistance Level')
ax.set_xlabel('AI Assistance Level')
ax.set_ylabel('Percentage of Sessions (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='upper right', title='Final Outcome')
plt.tight_layout()
plt.show()


### RQ3: Overreliance

In [ ]:
from src.statistical_analysis import rq3_overreliance

r3 = rq3_overreliance(df)
print(f"Struggle rate -- High AI: {r3['struggle_rate_high_AI']:.1%}")
print(f"Struggle rate -- Low AI:  {r3['struggle_rate_low_AI']:.1%}")
print(f"Chi-square: chi2({r3['dof']}) = {r3['chi2']}, p={r3['p']}")
print()
print('Gave-Up rate by level:')
for lvl, rate in r3['gaveup_by_level'].items():
    print(f'  Level {lvl}: {rate:.1%}')
print()
print('Interpretation:', r3['interpretation'])


### RQ4: Task Type Effects

In [ ]:
from src.statistical_analysis import rq4_task_type_effects

r4 = rq4_task_type_effects(df)
print(f"ANOVA: F={r4['anova_f']}, p={r4['anova_p']} {r4['anova_sig']}")
print()
sat_data = r4['satisfaction_by_task']
for task, data in r4['success_by_task'].items():
    sat = sat_data['mean_satisfaction'].get(task, float('nan'))
    print(f"  {task:<20} success={data['success_rate']:.1%}  sat={sat:.2f}")


In [ ]:
heat = (
    df.groupby(['TaskType', 'ai_level_group'])['SatisfactionRating']
    .mean().unstack()
)
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Satisfaction'})
ax.set_title('Satisfaction by Task Type x AI Level Group')
plt.tight_layout()
plt.show()


### RQ5: Beginners vs Experts

In [ ]:
from src.statistical_analysis import rq5_beginner_vs_expert

r5 = rq5_beginner_vs_expert(df)
print('Success rates:')
for lvl, rate in r5['success_by_level'].items():
    print(f'  {lvl}: {rate:.1%}')
tt = r5['success_ttest_hs_vs_grad']
print(f"\nHS vs Grad: t={tt['t']}, p={tt['p']}, Cohen's d={tt['d']} ({tt['effect_label']})")
print('\nInterpretation:', r5['interpretation'])


### RQ6: The Satisfaction Inflation Effect

> **Key finding:** AI level is strongly correlated with satisfaction (r=0.776, p<.001, R2=0.60) but satisfaction shows NO significant relationship with actual task success. This decoupling is the study's central contribution.

In [ ]:
from src.statistical_analysis import rq6_satisfaction_correlates

r6 = rq6_satisfaction_correlates(df)
print(f"r(Satisfaction, AI Level) = {r6['r_sat_ai']:.3f},  p = {r6['p_sat_ai']}")
print(f"r(Satisfaction, Success)  = {r6['r_sat_success']:.3f},  p = {r6['p_sat_success']}")
print(f"r(Satisfaction, Prompts)  = {r6['r_sat_prompts']:.3f},  p = {r6['p_sat_prompts']}")
print()
ols = r6.get('ols', {})
if ols:
    print(f"OLS R2 = {ols['r_squared']}, adj-R2 = {ols['adj_r_squared']}")
    print('Coefficients:')
    for param, coef in ols['params'].items():
        pval = ols['pvalues'][param]
        print(f'  {param:<30} coef={coef:>8.4f}  p={pval:.4f}')
print()
print('INTERPRETATION:', r6['interpretation'])


In [ ]:
# KEY CHART: Satisfaction Inflation Effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
jitter = np.random.default_rng(42).uniform(-0.25, 0.25, size=len(df))
axes[0].scatter(df['AI_AssistanceLevel'] + jitter, df['SatisfactionRating'],
                alpha=0.06, s=8, color='#4C72B0', rasterized=True)
slope, intercept, r, p, _ = stats.linregress(
    df['AI_AssistanceLevel'], df['SatisfactionRating'])
x_line = np.linspace(1, 5, 100)
axes[0].plot(x_line, intercept + slope * x_line, color='#C44E52', lw=2.5,
             label=f'r = {r:.3f} (p < .001)')
axes[0].set_title('AI Assistance Level vs Satisfaction')
axes[0].set_xlabel('AI Assistance Level (jittered)')
axes[0].set_ylabel('Satisfaction Rating (1-5)')
axes[0].set_xticks(range(1, 6))
axes[0].legend()

succ_means = df.groupby('success')['SatisfactionRating'].mean()
axes[1].bar(['Struggle (0)', 'Success (1)'], succ_means,
            color=['#CD5C5C', '#2E8B57'], edgecolor='white')
axes[1].set_title('Mean Satisfaction by Task Success')
axes[1].set_ylabel('Mean Satisfaction')
for i, v in enumerate(succ_means):
    axes[1].text(i, v + 0.04, f'{v:.3f}', ha='center', fontweight='bold')
plt.suptitle('The Satisfaction Inflation Effect', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### RQ7: Predictive Model

In [ ]:
from src.statistical_analysis import rq7_predictive_model

r7 = rq7_predictive_model(df)
print(f"5-fold CV AUC = {r7['cv_roc_auc_mean']:.3f} (SD={r7['cv_roc_auc_std']:.3f})")
print()
print('Feature coefficients (standardized):')
for feat, coef in r7['coefficients'].items():
    print(f'  {feat:<25} {coef:+.4f}')
print()
print('Note: AUC near 0.5 -- session features are weak predictors of individual struggle.')
print('Interpretation:', r7['interpretation'])


### RQ8: Repeat Usage

In [ ]:
from src.statistical_analysis import rq8_repeat_usage

r8 = rq8_repeat_usage(df)
print(f"UsedAgain rate: {r8['used_again_rate']:.1%}")
print(f"r(UsedAgain, Satisfaction): {r8['r_used_satisfaction']:.3f}  p={r8['p_used_satisfaction']}")
print(f"r(UsedAgain, Success):      {r8['r_used_success']:.3f}  p={r8['p_used_success']}")
print()
print('UsedAgain rate by outcome:')
for outcome, rate in r8['used_again_by_outcome'].items():
    print(f'  {outcome:<25}: {rate:.1%}')
print('\nInterpretation:', r8['interpretation'])


## 4. Publication Charts

In [ ]:
import matplotlib
matplotlib.use('Agg')
from src.visualization import generate_all_charts

paths = generate_all_charts(df)
print('Generated charts:')
for name, path in paths.items():
    print(f'  {name}: {path}')


## 5. Summary of Findings

| RQ | Finding | Effect | Significant? |
|----|---------|--------|--------------|
| RQ1: Session efficiency | No time difference by AI level | d = 0.015 | No |
| RQ2: Task success | 76.3% overall; AI level not predictive | V = 0.019 | No |
| RQ3: Overreliance | Struggle equal across AI groups (23%) | negligible | No |
| RQ4: Task type | No satisfaction variation by task type | F = 1.05 | No |
| RQ5: Beginner vs expert | Trivial performance gap (d=0.021) | negligible | No |
| **RQ6: Satisfaction inflation** | **AI level strongly predicts satisfaction r=0.776** | **R2=0.60** | **Yes ***  |
| RQ7: Struggle prediction | CV AUC = 0.534 (near-chance) | -- | Weak |
| **RQ8: Repeat usage** | **Success predicts UsedAgain r=0.371** | -- | **Yes ***  |

### Central Insight
Students who use AI more heavily report substantially higher session satisfaction, but this satisfaction is **decoupled from actual task success**. This satisfaction inflation effect is a hallmark pattern of potential overreliance risk.

In [ ]:
print('=' * 55)
print('  STUDY SUMMARY STATISTICS')
print('=' * 55)
print(f'  Total sessions analyzed:    {len(df):,}')
print(f"  Date range:                 {df['SessionDate'].min().date()} to {df['SessionDate'].max().date()}")
print(f"  Student levels:             {df['StudentLevel'].nunique()} (HS, UG, Grad)")
print(f"  Disciplines:                {df['Discipline'].nunique()}")
print(f"  Task types:                 {df['TaskType'].nunique()}")
print(f"  Overall success rate:       {df['success'].mean():.1%}")
print(f"  Would use AI again:         {df['UsedAgain'].mean():.1%}")
print(f"  Mean session length:        {df['SessionLengthMin'].mean():.1f} min")
print(f"  Mean prompts/session:       {df['TotalPrompts'].mean():.1f}")
print(f"  Mean AI assistance level:   {df['AI_AssistanceLevel'].mean():.2f} / 5")
print(f"  Mean satisfaction:          {df['SatisfactionRating'].mean():.2f} / 5")
print()
print('  KEY RESULTS:')
print('  r(Satisfaction ~ AI Level):   0.776 (p < .001)')
print('  r(UsedAgain ~ Success):       0.371 (p < .001)')
print('  OLS R-squared (Sat model):    0.602')
print('  Logistic CV AUC (Struggle):   0.534')
print('=' * 55)
